## **03 - Scraping Booking.com (Playwright)**


### **Pourquoi Playwright (et pas `requests` + `bs4`, ni `scrapy`)?**

Booking charge une partie du contenu en JavaScript et applique des mecanismes anti-bot.

- `requests` + `BeautifulSoup`: rapide, mais fragile si le HTML initial ne contient pas toutes les donnees rendues par JS.
- `scrapy`: excellent pour du crawl a grande echelle, mais ici il faut d'abord fiabiliser le rendu navigateur et les interactions.
- `playwright`: pilote un vrai navigateur (Chromium), rend le JavaScript, permet d'attendre les bons selecteurs et de gerer les interactions.

Conclusion: pour un scraping cible et robuste sur Booking, Playwright est le meilleur compromis.

> Note legale: respecte toujours les CGU du site cible, limite la frequence des requetes et evite toute surcharge.

In [20]:
import re
import asyncio
import pandas as pd
import plotly.express as px

from datetime import date, timedelta
from pathlib import Path
from urllib.parse import quote_plus
from playwright.async_api import (
    async_playwright,
    TimeoutError as PlaywrightTimeoutError,
    Error as PlaywrightError,
)

In [9]:
# Parametres de scraping
MAX_HOTELS_PER_CITY = 6
TOP_DESTINATIONS = 10
FINAL_TOP_HOTELS = 20
MAX_GOTO_RETRIES = 3
RETRY_BASE_DELAY_SECONDS = 2
HEADLESS = True
DEBUG_SCRAPE = True
DEFAULT_COUNTRY = "France"
CHECKIN_DATE = (date.today()).isoformat()
CHECKOUT_DATE = (date.today() + timedelta(days=7)).isoformat()

scored_path = Path("../outputs/data/cities_weather_scored.csv")
if not scored_path.exists():
    raise FileNotFoundError(
        "Fichier introuvable: ../outputs/data/cities_weather_scored.csv. Execute d'abord le notebook 02."
    )

scored_df = pd.read_csv(scored_path)

if "city_name" not in scored_df.columns:
    raise ValueError("La colonne 'city_name' est requise dans cities_weather_scored.csv")

if "country" not in scored_df.columns:
    scored_df["country"] = DEFAULT_COUNTRY

if "weather_rank" in scored_df.columns:
    top_destinations_df = scored_df.nsmallest(TOP_DESTINATIONS, "weather_rank").copy()
else:
    # Fallback: on garde les TOP_DESTINATIONS premieres lignes si le rang n'est pas present
    top_destinations_df = scored_df.head(TOP_DESTINATIONS).copy()

top_destinations_df[["city_name", "country"]]

,city_name,country
0,Carcassonne,France
1,Paris,France
2,Besancon,France
3,Nimes,France
4,Grenoble,France
5,Strasbourg,France
6,Colmar,France
7,Toulouse,France
8,Mont Saint Michel,France
9,Eguisheim,France


In [10]:
def build_booking_search_url(city: str, country: str, checkin: str, checkout: str) -> str:
    query = quote_plus(f"{city}, {country}")
    return (
        "https://www.booking.com/searchresults.fr.html"
        f"?ss={query}"
        f"&checkin={checkin}&checkout={checkout}"
        "&group_adults=2&no_rooms=1&group_children=0"
    )


def parse_lat_lon_from_html(html: str):
    atlas_match = re.search(r'data-atlas-latlng="([0-9\.-]+),([0-9\.-]+)"', html)
    if atlas_match:
        return float(atlas_match.group(1)), float(atlas_match.group(2))

    json_match = re.search(r'"latitude"\s*:\s*([0-9\.-]+).*?"longitude"\s*:\s*([0-9\.-]+)', html, re.S)
    if json_match:
        return float(json_match.group(1)), float(json_match.group(2))

    return None, None


async def extract_listing_cards(page):
    selectors = [
        'div[data-testid="property-card"]',
        'div[data-testid="property-card-container"]',
    ]
    for selector in selectors:
        cards = page.locator(selector)
        if await cards.count() > 0:
            return cards
    return page.locator('article')


async def safe_goto(page, url: str, city: str, label: str = "listing"):
    last_error = None
    for attempt in range(1, MAX_GOTO_RETRIES + 1):
        try:
            await page.goto(url, wait_until="domcontentloaded", timeout=90_000)
            return
        except (PlaywrightTimeoutError, PlaywrightError) as e:
            last_error = e
            msg = str(e)
            is_retryable = (
                "ERR_NETWORK_CHANGED" in msg
                or "ERR_TIMED_OUT" in msg
                or isinstance(e, PlaywrightTimeoutError)
            )
            if attempt >= MAX_GOTO_RETRIES or not is_retryable:
                raise
            wait_s = RETRY_BASE_DELAY_SECONDS * attempt
            print(f"  -> Retry {label} {attempt}/{MAX_GOTO_RETRIES} pour {city} ({wait_s}s): {e}")
            await asyncio.sleep(wait_s)

    if last_error:
        raise last_error


In [11]:
async def scrape_city_hotels(page, city: str, country: str, max_hotels: int = 5):
    search_url = build_booking_search_url(city, country, CHECKIN_DATE, CHECKOUT_DATE)
    await safe_goto(page, search_url, city=city, label="listing")
    await asyncio.sleep(2)

    # Accepte le bandeau cookies si present
    for cookie_selector in [
        'button#onetrust-accept-btn-handler',
        'button:has-text("J\'accepte")',
        'button:has-text("Accept")',
    ]:
        btn = page.locator(cookie_selector).first
        if await btn.count() > 0:
            try:
                await btn.click(timeout=2_000)
                break
            except Exception:
                pass

    try:
        await page.wait_for_selector('div[data-testid="property-card"]', timeout=20_000)
    except (PlaywrightTimeoutError, asyncio.CancelledError):
        pass

    page_html = await page.content()
    lower_html = page_html.lower()
    if "captcha" in lower_html or "robot" in lower_html or "verify you are human" in lower_html:
        print(f"  -> Blocage anti-bot detecte pour {city}")

    cards = await extract_listing_cards(page)
    hotels = []
    cards_count = await cards.count()

    if cards_count == 0 and DEBUG_SCRAPE:
        debug_dir = Path("../outputs/debug")
        debug_dir.mkdir(parents=True, exist_ok=True)
        safe_city = re.sub(r"[^a-zA-Z0-9_-]+", "_", city)
        screenshot_path = debug_dir / f"booking_{safe_city}.png"
        await page.screenshot(path=str(screenshot_path), full_page=True)
        print(f"  -> 0 carte trouvee. URL finale: {page.url}")
        print(f"  -> Screenshot debug: {screenshot_path.resolve()}")

    for i in range(min(cards_count, max_hotels)):
        card = cards.nth(i)

        title_locator = card.locator('div[data-testid="title"]').first
        name = await title_locator.inner_text(timeout=3_000) if await title_locator.count() else None

        link_el = card.locator('a[data-testid="title-link"]').first
        if await link_el.count() == 0:
            link_el = card.locator("a").first
        hotel_url = await link_el.get_attribute("href") if await link_el.count() else None
        if hotel_url and hotel_url.startswith("/"):
            hotel_url = f"https://www.booking.com{hotel_url}"

        score_text = None
        score_selectors = [
            'div[data-testid="review-score"]',
            'div[aria-label*="Scored"]',
            'div[aria-label*="Note"]',
        ]
        for s in score_selectors:
            score_el = card.locator(s).first
            if await score_el.count() > 0:
                score_text = await score_el.inner_text(timeout=2_000)
                break

        lat, lon, description = None, None, None
        if hotel_url:
            detail = await page.context.new_page()
            try:
                await safe_goto(detail, hotel_url, city=city, label="detail")
                detail_html = await detail.content()
                lat, lon = parse_lat_lon_from_html(detail_html)

                desc_selectors = [
                    '#property_description_content',
                    'p[data-testid="property-description"]',
                    'div[data-testid="property-description"]',
                ]
                for ds in desc_selectors:
                    desc_el = detail.locator(ds).first
                    if await desc_el.count() > 0:
                        description = (await desc_el.inner_text(timeout=3_000)).strip()
                        break
            except Exception as e:
                if DEBUG_SCRAPE:
                    print(f"  -> Detail incomplet pour {city}: {e}")
            finally:
                await detail.close()

        # On garde uniquement les enregistrements complets
        is_complete = all([
            name,
            hotel_url,
            score_text,
            description,
            lat is not None,
            lon is not None,
        ])
        if is_complete:
            hotels.append(
                {
                    "city_name": city,
                    "country": country,
                    "hotel_name": name,
                    "booking_url": hotel_url,
                    "latitude": lat,
                    "longitude": lon,
                    "score_text": score_text,
                    "description": description,
                }
            )

        await asyncio.sleep(1.2)

    return hotels

In [12]:
async def run_booking_scrape(top_destinations_df: pd.DataFrame) -> pd.DataFrame:
    all_hotels = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS, slow_mo=50)
        context = await browser.new_context(
            locale="fr-FR",
            user_agent=(
                "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
            ),
            viewport={"width": 1366, "height": 900},
        )
        page = await context.new_page()

        for row in top_destinations_df.itertuples(index=False):
            city = row.city_name
            country = row.country
            print(f"Scraping: {city}, {country}")

            try:
                city_hotels = await scrape_city_hotels(
                    page,
                    city=city,
                    country=country,
                    max_hotels=MAX_HOTELS_PER_CITY,
                )
                all_hotels.extend(city_hotels)
                print(f"  -> {len(city_hotels)} hotels recuperes")
            except asyncio.CancelledError:
                print(f"  -> Scraping annule pour {city}")
            except Exception as e:
                print(f"  -> Echec sur {city}: {e}")

        await context.close()
        await browser.close()

    return pd.DataFrame(all_hotels)


hotels_df = await run_booking_scrape(top_destinations_df)
hotels_df.head()

Scraping: Carcassonne, France
  -> Blocage anti-bot detecte pour Carcassonne
  -> Retry detail 1/3 pour Carcassonne (2s): Page.goto: Timeout 90000ms exceeded.
Call log:
  - navigating to "https://www.booking.com/hotel/fr/inter-les-oliviers.fr.html?aid=304142&label=gen173nr-10CAQoggJCGnNlYXJjaF9jYXJjYXNzb25uZSwgZnJhbmNlSA1YBGiRAYgBAZgBM7gBGcgBDNgBA-gBAfgBAYgCAagCAbgC6c2lzwbAAgHSAiQyMjQ1ZDJjNS04M2I2LTQyNzEtOThhNi03YzdmMTgxNzMyZWTYAgHgAgE&ucfs=1&arphpl=1&checkin=2026-04-23&checkout=2026-04-30&group_adults=2&req_adults=2&no_rooms=1&group_children=0&req_children=0&hpos=2&hapos=2&sr_order=popularity&srpvid=b49102f4fed30268&srepoch=1776903921&all_sr_blocks=5518311_376292185_2_33_0&highlighted_blocks=5518311_376292185_2_33_0&matching_block_id=5518311_376292185_2_33_0&sr_pri_blocks=5518311_376292185_2_33_0__82425&from_sustainable_property_sr=1&from=searchresults", waiting until "domcontentloaded"

  -> 6 hotels recuperes
Scraping: Paris, France
  -> Blocage anti-bot detecte pour Paris
  -> Retr

,city_name,country,hotel_name,booking_url,latitude,longitude,score_text,description
0,Carcassonne,France,Tribe Carcassonne,https://www.booking.com/hotel/fr/des-trois-cou...,43.210306,2.357756,"Avec une note de 8,1\n8,1\nTrès bien\n3 108 ex...",Bénéficiant d’un emplacement privilégié dans l...
1,Carcassonne,France,ibis Styles Carcassonne La Cité,https://www.booking.com/hotel/fr/inter-les-oli...,43.207584,2.372714,"Avec une note de 8,5\n8,5\nTrès bien\n2 815 ex...",L'ibis Styles Carcassonne La Cité est situé à ...
2,Carcassonne,France,Hotel du Roi & Spa by SOWELL COLLECTION,https://www.booking.com/hotel/fr/soleilvacance...,43.210850,2.357769,"Avec une note de 8,4\n8,4\nTrès bien\n1 594 ex...",L’établissement Hotel du Roi & Spa by SOWELL C...
3,Carcassonne,France,SOWELL HOTELS Les Chevaliers,https://www.booking.com/hotel/fr/soleil-vacanc...,43.211032,2.358200,"Avec une note de 8,2\n8,2\nTrès bien\n2 726 ex...","Situé à Carcassonne, à 2 minutes à pied de l'A..."
4,Carcassonne,France,Hotel de la Cité Carcassonne - MGallery Collec...,https://www.booking.com/hotel/fr/hotel-de-la-c...,43.205627,2.362696,"Avec une note de 9,0\n9,0\nFabuleux \n1 104 ex...",L’Hotel de la Cité & Spa MGallery vous accueil...


In [13]:
# Nettoyage leger du score numerique
hotels_df["score"] = (
    hotels_df["score_text"]
    .astype(str)
    .str.extract(r"([0-9]+[\.,]?[0-9]*)", expand=False)
    .str.replace(",", ".", regex=False)
)
hotels_df["score"] = pd.to_numeric(hotels_df["score"], errors="coerce")

hotels_df = hotels_df[
    [
        "city_name",
        "country",
        "hotel_name",
        "latitude",
        "longitude",
        "score",
        "description",
        "booking_url",
    ]
]

# On retire les lignes encore incompletes (double securite)
required_columns = ["hotel_name", "booking_url", "latitude", "longitude", "score", "description"]
hotels_df = hotels_df.dropna(subset=required_columns).copy()
hotels_df = hotels_df.drop_duplicates(subset=["booking_url"]).copy()

# Resultat final: top 20 hotels globaux complets (meilleure note d'abord)
hotels_df = hotels_df.sort_values("score", ascending=False, na_position="last").head(FINAL_TOP_HOTELS)

if len(hotels_df) < FINAL_TOP_HOTELS:
    raise ValueError(
        f"Seulement {len(hotels_df)} hotels complets recuperes. "
        f"Augmente TOP_DESTINATIONS ou MAX_HOTELS_PER_CITY, puis relance."
    )

hotels_df.sample(min(10, len(hotels_df)))

,city_name,country,hotel_name,latitude,longitude,score,description,booking_url
45,Toulouse,France,The Social Hub Toulouse,43.610053,1.430370,8.7,L’établissement The Social Hub Toulouse bénéfi...,https://www.booking.com/hotel/fr/the-social-hu...
22,Nimes,France,Maison Albar Hotels L’Imperator,43.838666,4.352768,9.1,La Maison Albar Hotels L’Imperator est un hôte...,https://www.booking.com/hotel/fr/imperator-nim...
32,Strasbourg,France,Hôtel Restaurant Athena Spa,48.591737,7.711705,8.7,Installé à moins de 3 km du centre-ville de St...,https://www.booking.com/hotel/fr/athena-spa.fr...
51,Mont Saint Michel,France,Gites Bellevue,48.607880,-1.517224,8.9,Situé à seulement 2 km du célèbre Mont-Saint-M...,https://www.booking.com/hotel/fr/gites-bellevu...
4,Carcassonne,France,Hotel de la Cité Carcassonne - MGallery Collec...,43.205627,2.362696,9.0,L’Hotel de la Cité & Spa MGallery vous accueil...,https://www.booking.com/hotel/fr/hotel-de-la-c...
57,Eguisheim,France,"James Vignoble Hôtel, Eguisheim",48.044963,7.301578,9.0,James Vignoble Hôtel Eguisheim is located amon...,https://www.booking.com/hotel/fr/james-vignobl...
28,Grenoble,France,"Le Grand Hôtel Grenoble, BW Premier Collection...",45.190815,5.728548,8.7,"L’établissement Le Grand Hôtel Grenoble, BW Pr...",https://www.booking.com/hotel/fr/le-grand-gren...
58,Eguisheim,France,Eguisheim village préféré des français grand s...,48.047683,7.313291,9.3,L’hébergement Eguisheim village préféré des fr...,https://www.booking.com/hotel/fr/studio-eguish...
7,Paris,France,Bradford Elysées - Astotel,48.872913,2.308191,9.1,"Occupant un bâtiment haussmannien, le Bradford...",https://www.booking.com/hotel/fr/bradfordelyse...
54,Eguisheim,France,Hotel SPA Husseren Collections - Proche Colmar...,48.034988,7.276138,8.9,"Entouré de vignobles et d’une forêt, l’Hotel S...",https://www.booking.com/hotel/fr/husseren-les-...


In [14]:
output_data_dir = Path("../outputs/data")
output_data_dir.mkdir(parents=True, exist_ok=True)

booking_output_file = output_data_dir / "top20_booking_hotels_enriched.csv"
hotels_df.to_csv(booking_output_file, index=False)

print(f"Export Booking: {booking_output_file.resolve()}")
print(f"Nombre de lignes (top final): {len(hotels_df)}")

Export Booking: /home/raphael/DataProjects/jedha-certif/projet-kayak-bloc1/outputs/data/top20_booking_hotels.csv
Nombre de lignes (top final): 20


## Visualisation carte des top 20 hotels

Carte **Plotly** (`scatter_mapbox`) avec fond **OpenStreetMap** (tuiles reelles), puis export PNG (via `kaleido`).

In [41]:
# Carte top 20 hotels : fond cartographique reel (OSM) + export PNG avec scatter_map (nouvelle API)
figures_dir = Path("../outputs/maps/")
figures_dir.mkdir(parents=True, exist_ok=True)

map_png_file = figures_dir / "top20_booking_hotels_map.png"

plot_df = hotels_df.copy()
plot_df["score"] = pd.to_numeric(plot_df["score"], errors="coerce")

center_lat = float(plot_df["latitude"].mean())
center_lon = float(plot_df["longitude"].mean())

fig = px.scatter_map(
    plot_df,
    lat="latitude",
    lon="longitude",
    color="score",
    hover_name="hotel_name",
    hover_data={
        "city_name": True,
        "country": True,
        "score": ":.1f",
        "latitude": ":.5f",
        "longitude": ":.5f",
    },
    color_continuous_scale="Viridis",
    zoom=5.2,
    height=780,
    title="Top 20 hotels Booking - France",
    center={"lat": center_lat, "lon": center_lon}
)

fig.update_traces(marker_size=14, marker_opacity=0.92)
fig.update_layout(
    margin=dict(l=0, r=0, t=50, b=0),
    legend_title_text="Score",
)

fig.show()

fig.write_image(str(map_png_file), width=1400, height=900, scale=2)
print(f"Carte exportee: {map_png_file.resolve()}")

Carte exportee: /home/raphael/DataProjects/jedha-certif/projet-kayak-bloc1/outputs/maps/top20_booking_hotels_map.png


## Installation (si necessaire)

```bash
pipenv install
pipenv run playwright install chromium
```

Puis execute ce notebook avec le kernel de l'environnement Pipenv.